In [ ]:
!wget https://github.com/karoldvl/ESC-50/archive/master.zip
!unzip master.zip

--2025-07-29 14:50:30--  https://github.com/karoldvl/ESC-50/archive/master.zip
Resolving github.com (github.com)... 140.82.121.4
Connecting to github.com (github.com)|140.82.121.4|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://github.com/karolpiczak/ESC-50/archive/master.zip [following]
--2025-07-29 14:50:31--  https://github.com/karolpiczak/ESC-50/archive/master.zip
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://codeload.github.com/karolpiczak/ESC-50/zip/refs/heads/master [following]
--2025-07-29 14:50:31--  https://codeload.github.com/karolpiczak/ESC-50/zip/refs/heads/master
Resolving codeload.github.com (codeload.github.com)... 140.82.121.10
Connecting to codeload.github.com (codeload.github.com)|140.82.121.10|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [application/zip]
Saving to: ‘master.zip’

master.zip              [   

In [ ]:
import pandas as pd

final_labels = {
    "fire_alarm": "fireworks",
    "siren": "siren",
    "dog_bark": "dog",
    "thunderstorm": "thunderstorm",
    "crying_baby": "crying_baby",
    "door_knock": "door_wood_knock",
    "glass_breaking": "glass_breaking",
    "horn": "car_horn",
    "smoke_beep": "smoke_alarm",
    "screaming": "screaming",
    "gun_shot": "gun_shot",
    "train": "train",
    "rain": "rain",
    "door_bell": "doorbell"
}

meta = pd.read_csv("ESC-50-master/meta/esc50.csv")
selected_categories = list(final_labels.values())
filtered_meta = meta[meta["category"].isin(selected_categories)]

print(f"Total filtered samples: {len(filtered_meta)}")
print("Filtered classes found:", filtered_meta['category'].unique())


Total filtered samples: 400
Filtered classes found: ['dog' 'thunderstorm' 'door_wood_knock' 'fireworks' 'train' 'car_horn'
 'rain' 'crying_baby' 'glass_breaking' 'siren']


In [ ]:
!wget https://zenodo.org/record/1203745/files/UrbanSound8K.tar.gz
!tar -xvzf UrbanSound8K.tar.gz

Streaming output truncated to the last 5000 lines.
UrbanSound8K/audio/fold4/17480-2-0-6.wav
UrbanSound8K/audio/fold4/17480-2-0-9.wav
UrbanSound8K/audio/fold4/175904-2-0-11.wav
UrbanSound8K/audio/fold4/175904-2-0-24.wav
UrbanSound8K/audio/fold4/176003-1-0-0.wav
UrbanSound8K/audio/fold4/176638-5-0-0.wav
UrbanSound8K/audio/fold4/177756-2-0-10.wav
UrbanSound8K/audio/fold4/177756-2-0-4.wav
UrbanSound8K/audio/fold4/177756-2-0-5.wav
UrbanSound8K/audio/fold4/177756-2-0-7.wav
UrbanSound8K/audio/fold4/179862-1-0-0.wav
UrbanSound8K/audio/fold4/180977-3-1-1.wav
UrbanSound8K/audio/fold4/180977-3-1-5.wav
UrbanSound8K/audio/fold4/183989-3-1-21.wav
UrbanSound8K/audio/fold4/183989-3-1-23.wav
UrbanSound8K/audio/fold4/185709-0-0-0.wav
UrbanSound8K/audio/fold4/185709-0-0-1.wav
UrbanSound8K/audio/fold4/185709-0-0-6.wav
UrbanSound8K/audio/fold4/185709-0-0-7.wav
UrbanSound8K/audio/fold4/185909-2-0-102.wav
UrbanSound8K/audio/fold4/185909-2-0-13.wav
UrbanSound8K/audio/fold4/185909-2-0-17.wav
UrbanSound8K/audio

###

In [ ]:
# !zip -r ESC-50.zip /content/ESC-50-master
# from google.colab import files
# files.download("ESC-50.zip")

In [ ]:
# !zip -r UrbanSound8K.zip /content/UrbanSound8K
# from google.colab import files
# files.download("UrbanSound8K.zip")

###

In [ ]:
import pandas as pd

# 14 final class names
final_labels = {
    "fire_alarm": "fireworks",
    "siren": "siren",
    "dog_bark": "dog",
    "thunderstorm": "thunderstorm",
    "crying_baby": "crying_baby",
    "door_knock": "door_wood_knock",
    "glass_breaking": "glass_breaking",
    "horn": "car_horn",
    "smoke_beep": "smoke_alarm",
    "screaming": "screaming",
    "gun_shot": "gun_shot",
    "train": "train",
    "rain": "rain",
    "door_bell": "doorbell"
}

esc_meta = pd.read_csv("ESC-50-master/meta/esc50.csv")
selected_esc = list(final_labels.values())

filtered_meta = esc_meta[esc_meta["category"].isin(selected_esc)]
filtered_meta = filtered_meta[filtered_meta["category"] != "gun_shot"]  # not in ESC-50
filtered_meta = filtered_meta[filtered_meta["category"] != "screaming"]
filtered_meta = filtered_meta[filtered_meta["category"] != "doorbell"]
filtered_meta = filtered_meta[filtered_meta["category"] != "smoke_alarm"]

print(f"Filtered ESC-50 samples: {len(filtered_meta)}")
print("ESC classes:", filtered_meta['category'].unique())


Filtered ESC-50 samples: 400
ESC classes: ['dog' 'thunderstorm' 'door_wood_knock' 'fireworks' 'train' 'car_horn'
 'rain' 'crying_baby' 'glass_breaking' 'siren']


In [ ]:
import librosa
import numpy as np
from tqdm import tqdm

X_esc = []
y_esc = []

for _, row in tqdm(filtered_meta.iterrows(), total=len(filtered_meta)):
    file_path = f"ESC-50-master/audio/{row['filename']}"
    label = row['category']

    try:
        # ✅ FIXED: Use keyword arguments for librosa
        signal, sr = librosa.load(file_path, sr=22050)
        mel = librosa.feature.melspectrogram(y=signal, sr=sr, n_mels=64)
        mel_db = librosa.power_to_db(mel, ref=np.max)

        if mel_db.shape[1] < 64:
            mel_db = np.pad(mel_db, ((0, 0), (0, 64 - mel_db.shape[1])), mode='constant')
        mel_db = mel_db[:, :64]

        X_esc.append(mel_db)
        y_esc.append(label)

    except Exception as e:
        print(f"Failed to process {file_path}: {e}")


100%|██████████| 400/400 [00:31<00:00, 12.67it/s]


In [ ]:
from tqdm import tqdm
import librosa
import numpy as np
import pandas as pd
import os

# Load metadata
urban_meta = pd.read_csv("UrbanSound8K/metadata/UrbanSound8K.csv")
gun_meta = urban_meta[urban_meta["class"] == "gun_shot"].head(40)

X_gun = []
y_gun = []

for _, row in tqdm(gun_meta.iterrows(), total=40):
    path = f"UrbanSound8K/audio/fold{row['fold']}/{row['slice_file_name']}"

    try:
        # ✅ FIXED: use keyword arguments
        signal, sr = librosa.load(path, sr=22050)
        mel = librosa.feature.melspectrogram(y=signal, sr=sr, n_mels=64)
        mel_db = librosa.power_to_db(mel, ref=np.max)

        if mel_db.shape[1] < 64:
            mel_db = np.pad(mel_db, ((0, 0), (0, 64 - mel_db.shape[1])), mode='constant')
        mel_db = mel_db[:, :64]

        X_gun.append(mel_db)
        y_gun.append("gun_shot")

    except Exception as e:
        print(f"Failed to process {path}: {e}")


100%|██████████| 40/40 [00:00<00:00, 62.75it/s]


In [ ]:
X = np.array(X_esc + X_gun)
y = np.array(y_esc + y_gun)

print("Combined dataset shape:", X.shape)
print("Labels:", np.unique(y))


Combined dataset shape: (440, 64, 64)
Labels: ['car_horn' 'crying_baby' 'dog' 'door_wood_knock' 'fireworks'
 'glass_breaking' 'gun_shot' 'rain' 'siren' 'thunderstorm' 'train']


In [ ]:
import numpy as np

# Combine features and labels
X_all = np.concatenate([X_esc, X_gun])
y_all = np.concatenate([y_esc, y_gun])

print("Combined X shape:", X_all.shape)
print("Combined y shape:", y_all.shape)
print("Classes in y_all:", np.unique(y_all))


Combined X shape: (440, 64, 64)
Combined y shape: (440,)
Classes in y_all: ['car_horn' 'crying_baby' 'dog' 'door_wood_knock' 'fireworks'
 'glass_breaking' 'gun_shot' 'rain' 'siren' 'thunderstorm' 'train']


In [ ]:
from sklearn.preprocessing import LabelEncoder
import json

# Fixed class order (same as your 14 final class list)
final_class_names = [
    "fireworks", "siren", "dog", "thunderstorm", "crying_baby",
    "door_wood_knock", "glass_breaking", "car_horn", "smoke_alarm",
    "screaming", "gun_shot", "train", "rain", "doorbell"
]

# Fit encoder
le = LabelEncoder()
le.fit(final_class_names)

# Encode current labels
y_encoded = le.transform(y_all)

# ✅ FIX: Convert NumPy int64 to Python int
label_map = {label: int(code) for label, code in zip(le.classes_, le.transform(le.classes_))}

# Save to JSON
with open("label_map.json", "w") as f:
    json.dump(label_map, f)

print("✅ Label map saved:", label_map)


✅ Label map saved: {np.str_('car_horn'): 0, np.str_('crying_baby'): 1, np.str_('dog'): 2, np.str_('door_wood_knock'): 3, np.str_('doorbell'): 4, np.str_('fireworks'): 5, np.str_('glass_breaking'): 6, np.str_('gun_shot'): 7, np.str_('rain'): 8, np.str_('screaming'): 9, np.str_('siren'): 10, np.str_('smoke_alarm'): 11, np.str_('thunderstorm'): 12, np.str_('train'): 13}


In [ ]:
X_ready = X.reshape(-1, 1, 64, 64).astype(np.float32)
y_ready = y_encoded.astype(np.int64)

print("Final input shape:", X_ready.shape)
print("Final label shape:", y_ready.shape)


Final input shape: (440, 1, 64, 64)
Final label shape: (440,)


###folder structure


In [ ]:
import os

base_dir = "mel_dataset"
os.makedirs(base_dir, exist_ok=True)

for label in np.unique(y_all):
    class_folder = os.path.join(base_dir, label)
    os.makedirs(class_folder, exist_ok=True)


In [ ]:
import matplotlib.pyplot as plt
import uuid  # to generate unique filenames

def save_spectrograms(X, y, save_path):
    for idx, (spec, label) in enumerate(zip(X, y)):
        label_folder = os.path.join(save_path, label)
        os.makedirs(label_folder, exist_ok=True)

        filename = f"{uuid.uuid4().hex}.png"  # unique name
        full_path = os.path.join(label_folder, filename)

        # Save spectrogram image
        plt.imsave(full_path, spec, cmap='gray')


In [ ]:
save_spectrograms(X_all, y_all, "mel_dataset")


In [ ]:
!zip -r mel_dataset.zip /content/mel_dataset
from google.colab import files
files.download("mel_dataset.zip")

  adding: content/mel_dataset/ (stored 0%)
  adding: content/mel_dataset/crying_baby/ (stored 0%)
  adding: content/mel_dataset/crying_baby/12b8c27c62764198ae9f7daaa953853e.png (deflated 3%)
  adding: content/mel_dataset/crying_baby/6cf692f52b304e11afe4071f98b79109.png (deflated 2%)
  adding: content/mel_dataset/crying_baby/2990b02bf6d94dc5a88018bddf98a097.png (deflated 3%)
  adding: content/mel_dataset/crying_baby/b5b58f6488234b099124df92058bad72.png (deflated 2%)
  adding: content/mel_dataset/crying_baby/8f0bf0e4276c4e4d87b08e181aa4345f.png (deflated 4%)
  adding: content/mel_dataset/crying_baby/add3a0e7b39f4de4817dad3090653a44.png (deflated 1%)
  adding: content/mel_dataset/crying_baby/f360e9f8951c4679a3b3a60491a77495.png (deflated 2%)
  adding: content/mel_dataset/crying_baby/e0f88f19ca6b404e9d85254c9e67241b.png (deflated 3%)
  adding: content/mel_dataset/crying_baby/1ff1128a0ac64db9b7f37c3f46708186.png (deflated 4%)
  adding: content/mel_dataset/crying_baby/5404e4eee5274b1b8d7b87c

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>